# 03 — Failure-case mining (B-11, B-12)

Plan: D-24 in `doc/DECISIONS.md`. **Quantitative selection first, visual inspection second** (spec §7: no cherry-picking).

- All comparisons use **direction-aligned percentile ranks** (higher = better), so metrics on different scales (and different formulas) are comparable.
- Excluded: the 320 pixel-identical pairs (D-18). Candidates with high rater disagreement (`mos_var` above the 90th percentile) are excluded from selection, though they are still ranked.
- Diversity caps: at most 1 case per distortion type and per reference image (case A also: at most 2 per metric pair).

Outputs: `results/failures/*.csv` (full candidate lists, auditable) and `results/figures/failures/*.png`.
**The final 6–8 cases and their explanations are chosen by humans** (cells at the end).

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from humper.data import REPO_ROOT
from humper import failures as F
from humper.viz import case_figure, pair_figure

OUT = REPO_ROOT / "results" / "failures"; OUT.mkdir(parents=True, exist_ok=True)
FIG = REPO_ROOT / "results" / "figures" / "failures"; FIG.mkdir(parents=True, exist_ok=True)
K, VAR_Q = 5, 0.9

scores = pd.read_csv(REPO_ROOT / "results" / "scores.csv")
ev = F.eval_set(scores)
print(f"{len(ev)} pairs after dropping identical; mos_var cut = {scores.mos_var.quantile(VAR_Q):.3f}")
R = F.ranks(ev).set_index(ev["id"])   # global percentile ranks, used for annotations
rows = ev.set_index("id", drop=False)

## Systematic vs isolated
Share of images per distortion type where a metric's rank is > 0.5 away from the MOS rank. High = the metric fails on the whole type (systematic); low = failures are content-specific.

In [ ]:
sysm = F.systematic(scores, threshold=0.5)
sysm.to_csv(OUT / "systematic.csv")
fig, ax = plt.subplots(figsize=(6, 7))
im = ax.imshow(sysm.values, cmap="Reds", vmin=0, vmax=max(0.3, sysm.values.max()))
ax.set_xticks(range(sysm.shape[1]), sysm.columns, rotation=45, ha="right"); ax.set_yticks(range(len(sysm)), sysm.index)
for (i, j), v in pd.DataFrame(sysm.values).stack().items():
    ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7)
fig.colorbar(im, ax=ax, shrink=0.6, label="share |rank gap| > 0.5"); fig.tight_layout()
fig.savefig(FIG / "systematic_heatmap.png", dpi=150)
sysm.sort_values("psnr", ascending=False).head(10)

## Case A — two metrics strongly disagree (same image)
`gap = |r_a − r_b|` over all 10 metric pairs.

In [ ]:
A = F.metric_pairs(scores, VAR_Q)
A.head(50).to_csv(OUT / "candidates_A.csv", index=False)
selA = F.select_diverse(A, K, "gap", {"dist_name": 1, "ref_id": 1, "pair": 2})
selA.to_csv(OUT / "selected_A.csv", index=False)
print("cross-family share of top-200 candidates:", A.head(200).cross_family.mean().round(2))
selA[["id", "dist_name", "level", "pair", "rank_a", "rank_b", "rank_mos", "gap"]].round(3)

## Case B — a metric disagrees with humans
Signed `gap = r_metric − r_MOS`. **Too optimistic** (> 0): the metric misses damage that people see. **Too pessimistic** (< 0): the metric penalises something people barely notice.

In [ ]:
B = F.vs_human(scores, VAR_Q)
B.head(50).to_csv(OUT / "candidates_B.csv", index=False)
caps = {"dist_name": 1, "ref_id": 1, "metric": 2}
selB_opt = F.select_diverse(B[B.direction == "too_optimistic"], K, "gap", caps)
selB_pes = F.select_diverse(B[B.direction == "too_pessimistic"], K, "gap", caps)
selB = pd.concat([selB_opt, selB_pes], ignore_index=True)
selB.to_csv(OUT / "selected_B.csv", index=False)
selB[["direction", "id", "dist_name", "level", "metric", "rank_metric", "rank_mos", "gap"]].round(3)

## Case C — traditional vs learned *ranking* (same reference image)
Pairs where the traditional family (PSNR, SSIM, MS-SSIM) and the learned family (LPIPS-VGG, DISTS) order two distorted versions of the same image oppositely. `strength = min(|Δtrad|, |Δlearned|)`. Who do humans side with? Selection skips human ties (|ΔMOS| < 0.5), since a tie can't show which family failed.

In [ ]:
C = F.family_pairs(scores, VAR_Q, min_mos_diff=0.5)
C.head(50).to_csv(OUT / "candidates_C.csv", index=False)
print(len(C), "conflicting pairs; humans side with (all / strength > 0.3):")
print(pd.concat([C.human_sides_with.value_counts(normalize=True).rename("all"),
                 C[C.strength > 0.3].human_sides_with.value_counts(normalize=True).rename("strong")], axis=1).round(3))
# select only pairs where humans clearly side with one family (|ΔMOS| >= 0.5): a tie cannot show who failed
selC = F.select_diverse(C[C.human_sides_with != "tie"], K, "strength", {"ref_id": 1, "types": 1})
selC.to_csv(OUT / "selected_C.csv", index=False)
selC[["ref_id", "id_i", "dist_i", "id_j", "dist_j", "d_trad", "d_learned", "d_mos", "strength", "human_sides_with"]].round(3)

In [ ]:
tau = F.family_agreement(scores)
tau.to_csv(OUT / "family_agreement_by_ref.csv", index=False)
print("references where traditional and learned rankings agree least:"); tau.head(5).round(3)

## Figures for every selected candidate
reference | distorted | |diff| | 1−SSIM map | LPIPS-VGG map; titles show every score and its rank `r`.

In [ ]:
for k, r in selA.iterrows():
    fig = case_figure(rows.loc[r.id], R.loc[r.id], f"A{k+1}: {r.pair} gap {r.gap:.2f}")
    fig.savefig(FIG / f"A{k+1}_{r.id}.png", dpi=130, bbox_inches="tight"); plt.close(fig)
for k, r in selB.iterrows():
    tag = "Bo" if r.direction == "too_optimistic" else "Bp"
    fig = case_figure(rows.loc[r.id], R.loc[r.id], f"{tag}{k % K + 1}: {r.metric} {r.direction} gap {r.gap:+.2f}")
    fig.savefig(FIG / f"{tag}{k % K + 1}_{r.id}.png", dpi=130, bbox_inches="tight"); plt.close(fig)
for k, r in selC.iterrows():
    note = f"C{k+1}: traditional prefers {'i' if r.d_trad > 0 else 'j'}, learned prefers {'i' if r.d_learned > 0 else 'j'}; humans side with {r.human_sides_with} (ΔMOS i−j = {r.d_mos:+.2f})"
    fig = pair_figure(rows.loc[r.id_i], rows.loc[r.id_j], note)
    fig.savefig(FIG / f"C{k+1}_{r.id_i}_vs_{r.id_j}.png", dpi=130, bbox_inches="tight"); plt.close(fig)
sorted(p.name for p in FIG.glob("[ABC]*.png"))

## Final selection (humans: @hwupu + @ALLEN07056)

Pick **6–8 cases**, at least 2 from each of A, B, C, from the candidates above. Record each pick and why here, and in `doc/DECISIONS.md`.

| Case | id(s) | Picked because | Observation (draft) |
|---|---|---|---|
| | | | |

*Explanations are ours, not the AI's (AGENTS.md). Possible factors: structure, texture, spatial localisation, contrast, color, blur, noise, semantic content, salient regions, small geometric changes.*